# B2.1 · Reading the repository: history, index, components, map

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.0 · What a harness is, and why you are the one building it](https://spbreed.github.io/cyber-commons/lessons/B2.0.html)**.

| | |
|---|---|
| Tools used | git, OpenGrep, tree-sitter, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Parse a repo's vulnerability history into risk zones, then build a function/class index and rank files by prior-defect density.

**Why a security engineer needs it.** Review starts at the diff, so the pipeline never learns which parts of the repo keep breaking. The control it builds is: stages 1–2: mine commit and advisory history for repeat risk zones, then index the codebase into semantic units.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

An agent with a two-million-token context and a four-million-line repository has the same problem as an analyst with a week: it cannot read everything, so the question is what it reads first. Structure is what makes that choice something other than luck.

> **At CyberTravels.** CyberTravels' repository is four services and a shared library. Structure is what decides whether the pipeline reads the booking handlers or the vendored SDK first.

## 2 · The framework

```
   4,000,000 lines            context budget: ~30,000 lines
   +-------------------+      +----------------+
   |  the repository   | ---> |  what it reads |
   +-------------------+      +----------------+
              |
        structure decides the arrow

   symbol graph . call edges . entrypoints . change history
   -> "start at the functions reachable from an HTTP handler and changed
      in the last year" is a choice. "the first 30k lines" is not.
```

Most review starts at the diff. That is the smallest possible context, and it
throws away the single best predictor you have: **this repository has already
told you where it breaks.**

Phase 1 is four stages, and they run before any analysis. They take a
repository and produce the one artefact everything downstream consumes — a map.

**Stage 1 — Historical parsing.** Extract prior vulnerabilities, the commits
that fixed them, and pull-request history. Files that have been fixed for
security reasons before are dramatically more likely to be fixed again. It is
one of the oldest empirical results in software engineering, and almost nobody
wires it into a scanner.

**Stage 2 — Structural indexing.** Break the code into *semantic units* —
functions, classes, modules — and index how they call each other. Not lines,
not files. A scanner that reasons over lines cannot answer "who reaches this?",
and every later stage needs that answer.

**Stage 3 — Component summarisation.** One short summary per directory: what it
is for, what it talks to, what data passes through it. *Local* is the important
word — summarise the whole repository at once and you get a paragraph that is
true of every repository.

**Stage 4 — Architecture synthesis.** Compile the summaries into a single map
carrying three things: **entry points** where untrusted input arrives, **data
flows** between components, and **trust boundaries** where data crosses from
less trusted to more trusted.

The map is the artefact. Stage 5 reads its boundaries, stage 7 prioritises
against them, stage 10 walks its flows. And because it is derived rather than
drawn, it changes when the code changes — which is the property the last cell
in this lesson demonstrates and the reason the next lesson works at all.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Stage 1 — historical parsing

A slice of the CyberTravels bookings repository: commits, their subjects, and which of them were security fixes. Nothing has been scanned yet.

## 4 · Stage 2 — structural indexing

Index the code into semantic units. `ast` does the work here; in a polyglot repository this is what tree-sitter is for.

## 5 · Stage 3, with a model actually doing the summarising

In [ ]:
# --- model backend: replay by default, a Kaggle open-weight model when served -
# One URL and one header shape, no vendor SDK. Standard library only, so the
# notebook stays self-contained.
import json, os, urllib.error, urllib.request

# Qwen2.5-7B-Instruct is the floor established in MODELS.md: below it two of
# the lessons' acceptance properties stop holding.
OPEN_WEIGHT_DEFAULT = "qwen2.5-7b-instruct"
TIMEOUT = 60

def backend():
    """(kind, model). Configuration comes from the environment, never a literal."""
    if os.environ.get("OPENAI_BASE_URL"):
        return "open-weight", os.environ.get("MODEL", OPEN_WEIGHT_DEFAULT)
    return "replay", "deterministic stand-in (no backend configured)"

def _post(url, payload, headers):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"content-type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        return json.loads(r.read().decode())

def _openai_compatible(prompt, system, model, max_tokens, temperature):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    base = os.environ["OPENAI_BASE_URL"].rstrip("/")
    key = os.environ.get("OPENAI_API_KEY", "not-needed")
    out = _post(f"{base}/chat/completions",
                {"model": model, "messages": msgs, "max_tokens": max_tokens,
                 "temperature": temperature},
                {"authorization": f"Bearer {key}"})
    return out["choices"][0]["message"]["content"].strip()

def ask(prompt, *, replay, system=None, max_tokens=512, temperature=0.0):
    """Answer `prompt` with the configured backend, or return `replay`.

    `replay` is required, not optional: a lesson must be able to run offline,
    and the answer it falls back to has to be visible in the source rather than
    invented at runtime.
    """
    kind, model = backend()
    if kind == "replay":
        return replay, kind, model
    try:
        return _openai_compatible(prompt, system, model, max_tokens,
                                  temperature), kind, model
    except (urllib.error.URLError, urllib.error.HTTPError, KeyError, TimeoutError) as e:
        # Print what the server actually said. "failed: 400" costs whoever hits
        # this an hour; the body usually names the exact missing parameter, and
        # it never contains a key.
        detail = getattr(e, "code", None) or type(e).__name__
        why = ""
        if hasattr(e, "read"):
            try:
                why = json.loads(e.read().decode()).get("error", {}).get("message", "")
            except Exception:
                why = ""
        print(f"   !! {kind} backend ({model}) failed: {detail}"
              f"{' - ' + why if why else ''}")
        print("      Using the replay, which is labelled as one. No model answered.")
        return replay, "replay", f"{model} unreachable"

_kind, _model = backend()
print(f"model backend : {_kind}")
print(f"model         : {_model}")
if _kind == "replay":
    print()
    print("This lesson runs offline against a deterministic replay, which is why")
    print("it works on a Kaggle kernel with the internet switched off. To run the")
    print("identical code against a real model, serve an open-weight model from")
    print("Kaggle Models and point the adapter at it:")
    print()
    print("   python3 -m llama_cpp.server --model <the .gguf from Kaggle> \\")
    print("           --model_alias qwen2.5-7b-instruct --port 11434 --chat_format qwen")
    print("   export OPENAI_BASE_URL=http://127.0.0.1:11434/v1 \\")
    print("          MODEL=qwen2.5-7b-instruct")
    print()
    print("   MODELS.md has the exact Kaggle download. There is no paid backend:")
    print("   every model result in this repository was produced this way.")

## 6 · The same lesson, against a real model

Everything below this point runs identically on two backends. Offline it uses a
deterministic replay that is labelled as a replay wherever it appears — never
presented as a model's output. With `OPENAI_BASE_URL` set it calls an
OpenAI-compatible server, which is how the open-weight models on Kaggle are
served — and how every model result in this repository was produced.

The point of running it both ways is not that the answers match. It is that
**the lesson's assertion holds either way** — if it only holds against the
replay, the lesson was testing the replay.

In [ ]:
TASK = 'Summarise what this component does in one sentence, then name its trust boundary.\n\nFiles: src/api/bookings.py\nExports: get_booking(request), upload_voucher(request)\nCallers: the public HTTP router. Calls into: src/data/reports.py, src/data/docs.py'

REPLAY = 'Accepts traveller HTTP requests for bookings and voucher uploads, passing both straight into the data layer.\nTrust boundary: every parameter here arrives from the public internet, so the edge between src/api and src/data is where untrusted input crosses into a component that reaches the database and the filesystem.'

answer, used, model = ask(TASK, replay=REPLAY,
            system='You summarise code components for a security architecture map. Two sentences, no preamble.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("names a trust boundary", "trust" in answer.lower() or "boundary" in answer.lower())
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, two possible backends. Offline the answer is")
print("the replay and is labelled as one; with a served model it is the model's.")

## 7 · Stage 3 — summarise each component, locally

The model call above is what stage 3 looks like in production. Below is the deterministic version, so the rest of the lesson has a fixed input to work from.

## 8 · Stage 4 — synthesise the map, and find the boundaries

A trust boundary is any edge where data crosses from a less-trusted component into a more-trusted one. Those edges are where every finding in the rest of the pipeline turns out to live.

## 9 · The map changes when the code changes

This is the whole reason for deriving it. One function is added; the map is regenerated; the delta is the thing the next lesson threat-models.

## 10 · Write the four stages down as an agent skill

You have just run Phase 1 by hand. The next repository needs the same four stages and so does the next agent, so the procedure belongs in a file rather than in your head. This is the one in this repository:

In [ ]:
# skills/appsec/appsec-repo-recon/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: appsec-repo-recon
description: >-
  Build the structural and historical map of a codebase before any security
  analysis. Use at the start of an application security review, when asked to
  find entry points, sinks, trust boundaries or attack surface, when triaging
  which files deserve attention, or when a later stage needs an architecture
  map it does not have.
allowed-tools: Read, Grep, Glob, Bash
---

# AppSec pipeline · Phase 1 — Ingestion and structural mapping

Covers **stages 1–4**. Produces the map every later stage reads. Nothing in
this phase decides whether the code is vulnerable — it decides *where to look*.

Most review starts at the diff, which is the smallest context available and
discards the best predictor there is: the repository has already recorded where
it breaks.

## When to use this

Load this skill first in any review of a codebase you have not mapped. If a
threat model, audit or report is requested and no `architecture_map` exists,
build one here rather than guessing from file names.

## Inputs

| Input | Required | Notes |
|---|---|---|
| Repository worktree | yes | read-only is sufficient |
| Commit history | preferred | degraded mode without it; say so in `caveats` |
| Issue/PR history | optional | improves stage 1 only |

## Procedure

**Stage 1 — Historical parsing.** Extract prior vulnerabilities, the commits
that fixed them, and their files. Security fixes cluster: a file patched for a
vulnerability once is materially more likely to hold another. Record a
`fix_count` per file. Never treat absence of history as evidence of safety —
it is usually evidence of a young file or a squashed import.

**Stage 2 — Structural indexing.** Enumerate units (functions, methods,
handlers) with file, line, parameters, and the calls each one makes. This is
the index every later stage joins against, so record identity as
`(file, unit)` — never a bare basename. Two `handler.py` files in different
directories are two different units, and collapsing them silently merges their
findings.

**Stage 3 — Component summarisation.** For each unit, record what it *touches*:
network, filesystem, database, subprocess, credentials, deserialisation. A unit
that touches none of these cannot be a sink, and excluding it early is the
cheapest correct filter in the pipeline.

**Stage 4 — Architecture synthesis.** Join the above into:
- **entry points** — units reachable from outside the trust boundary
- **sinks** — units that touch a dangerous resource
- **flows** — the call edges connecting them
- **trust boundaries** — the edges where the caller's trust level drops

Then compute reachability: for every entry point, which sinks can it reach.
An unreachable sink is not an attack surface, and a reachable one is the whole
list for Phase 2.

## Output contract

Emit exactly this shape. Later phases join on these keys.

```json
{
  "architecture_map": {
    "entry_points": [{"unit": "str", "file": "str", "line": 0, "exposure": "public|authenticated|internal"}],
    "sinks":        [{"unit": "str", "file": "str", "resource": "network|filesystem|database|subprocess|credential|deserialisation"}],
    "flows":        [["caller_unit", "callee_unit"]],
    "boundaries":   [{"edge": "a → b", "from_trust": 0, "to_trust": 0}],
    "reachable":    [{"entry": "str", "sink": "str", "path": ["unit", "..."]}],
    "hotspots":     [{"file": "str", "fix_count": 0}],
    "caveats":      ["str"]
  }
}
```

Order every list deterministically — sort by a full key, never rely on set or
dict iteration order. Two runs of this skill over the same commit must produce
byte-identical output, or the diff between two scans is meaningless.

## Failure modes

- **Reporting a sink with no path from an entry point.** That is a code smell,
  not an attack surface. Keep it out of `reachable`.
- **Matching paths by basename.** Match on parent directory plus filename tail.
- **Claiming completeness.** If the index skipped a language, generated code,
  or a vendored tree, list it in `caveats`. A silently partial map is worse
  than a small one, because the next stage cannot tell.

## Handoff

Pass `architecture_map` to **appsec-threat-model**. If `reachable` is empty,
stop and report that — do not proceed to threat modelling on an empty surface.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/appsec/appsec-repo-recon/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

## What you just proved

Four of ten commits match the security markers, and `src/api/bookings.py` ranks highest on decayed risk purely from history — with the most recent security commit scoring nothing, because it matches no marker. The structural index extracts five functions and identifies two entry points. Component summaries name what each directory touches, and the synthesised map shows both entry points reaching the database and the filesystem across a trust boundary. Adding one function adds a third entry point and two more boundary crossings.

## Your turn

Run stage 1 against a repository you own: `git log --name-only --grep='CVE\|security\|injection'`, then rank the files by how often they appear. That list usually surprises people, and it is free. Then check how many of your recent security fixes your markers would have missed.

---

**Next → [B2.2 · Threat modelling from what the estate already knows](https://spbreed.github.io/cyber-commons/lessons/B2.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*